# Assignment 1: Question 4

In [ ]:

import heapq
import random
from math import inf

# Simple directed graph: node -> list of (neighbor, travel_time)
graph = {
    "S": [("A", 4), ("B", 2)],
    "A": [("C", 3), ("P1", 6)],
    "B": [("A", 1), ("P2", 5)],
    "C": [("P1", 2), ("P2", 2)],
    "P1": [("D", 1)],
    "P2": [("D", 1)],
    "D": []
}

alpha = 1.0   # weight for parking cost
beta = 0.01   # weight per meter of walking

def dijkstra(graph, start):
    dist = {node: inf for node in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist[u]:
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist.get(v, inf):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

def objective(dist_from_start, parking_lots, lot_id, alpha=1.0, beta=0.01):
    info = parking_lots[lot_id]
    # constraints
    if (not info["available"]) or (info["walk_dist"] > info["max_walk"]):
        return inf
    drive_time = dist_from_start.get(lot_id, inf)
    if drive_time == inf:
        return inf
    return drive_time + alpha * info["cost"] + beta * info["walk_dist"]

def hill_climb_choose_lot(dist_from_start, parking_lots, alpha=1.0, beta=0.01, start_lot=None):
    lots = list(parking_lots.keys())
    current = start_lot if start_lot in parking_lots else random.choice(lots)
    current_score = objective(dist_from_start, parking_lots, current, alpha, beta)

    improved = True
    while improved:
        improved = False
        for cand in lots:
            if cand == current:
                continue
            cand_score = objective(dist_from_start, parking_lots, cand, alpha, beta)
            if cand_score < current_score:
                current, current_score = cand, cand_score
                improved = True
    return current, current_score



Test 1: Both Parking Lots Feasible

In [ ]:


max_walk = 500
parking_lots = {
    "P1": {"cost": 8, "walk_dist": 350, "available": True,  "max_walk": max_walk},
    "P2": {"cost": 3, "walk_dist": 450, "available": True,  "max_walk": max_walk},
}

dist = dijkstra(graph, "S")
best_lot, best_score = hill_climb_choose_lot(dist, parking_lots, alpha, beta)

print("Distances from S:", dist)
print("Hill Climbing chose:", best_lot)
print("Objective value:", best_score)


Distances from S: {'S': 0, 'A': 3, 'B': 2, 'C': 6, 'P1': 8, 'P2': 7, 'D': 8}
Hill Climbing chose: P2
Objective value: 14.5


Test 2: Walking Constraint Eliminates P2

In [ ]:

parking_lots2 = {
    "P1": {"cost": 8, "walk_dist": 350, "available": True,  "max_walk": 500},
    "P2": {"cost": 3, "walk_dist": 650, "available": True,  "max_walk": 500},  # too far now
}

dist = dijkstra(graph, "S")
best_lot, best_score = hill_climb_choose_lot(dist, parking_lots2, alpha, beta)

print("Hill Climbing chose:", best_lot)
print("Objective value:", best_score)


Test 2: P2 violates walking constraint
Hill Climbing chose: P1
Objective value: 19.5
